# Getting Started / Data and Knowledge Resolution
A run reads four things, each through a **resolver** (`CLIENT-RESOLVERS`): the patients' data, the value sets, the
CQL libraries, and results already computed. The environment gives every role a default, its tables. Each role can
come from somewhere else too, and the rest stay the environment's.

| Role | The environment's (default) | Files in a Volume | In the notebook |
|---|---|---|---|
| Data | the `data` tables | `VolumeDataResolver`: `<Type>.ndjson` files | `DataFrameDataResolver`: a DataFrame per type |
| Terminology | `knowledge.valueset` | `FolderTerminologyResolver`: FHIR `ValueSet` JSON | `DataFrameTerminologyResolver` |
| Knowledge | `knowledge.library` and `.measure` | `FolderKnowledgeResolver`: `.cql` files, in a Volume or a workspace folder | `%%cql` cells |
| Results | your own, then the official ones | — | `SessionResultStore`: this session's temporary tables |

The deploy writes the same content as tables and as files, in each schema's `files` volume, so every path below
gives the same answer.

**Runs on serverless — no cluster to attach.**

In [ ]:
%reload_ext cql_databricks.compiler
from cql_databricks.compiler import cql

In [ ]:
# SVC-EXAMPLE-RESOLUTION-PATHS — the volumes are the default environment's, found from its catalog.
# Start clean, so Run All gives the same output however often it runs (CLIENT-SESSION-RESET).
import json
import os

from cql_databricks.compiler import reset, use_resolvers
from cql_databricks.data_bindings import spark_runner
from cql_databricks.resolvers import EnvironmentResolver

reset(spark)
environment = EnvironmentResolver(spark_runner(spark)).default()
DATA_FILES = f"/Volumes/{environment.catalog}/data/files"
KNOWLEDGE_FILES = f"/Volumes/{environment.catalog}/knowledge/files"
print("environment:", environment.name, "in catalog", environment.catalog)
for folder in (f"{DATA_FILES}/fhir", f"{KNOWLEDGE_FILES}/valuesets", f"{KNOWLEDGE_FILES}/libraries"):
    print(f"{folder}: {len(os.listdir(folder))} files")

**Expected output:** the environment, then 6 FHIR files, 20 value sets and 6 libraries.

## 1. Data and terminology: three paths, one answer
Who has a colonoscopy on record? The `%%cql` cell declares the question and reads nothing; each Python cell after it
runs it over a different source. `use_resolvers` sets the sources for this notebook's `cql()` calls and cells; a role
it is not given is the environment's, and `use_resolvers()` with nothing goes back to the environment.

In [ ]:
%%cql
using FHIR version '4.0.1'

valueset "Colonoscopy": 'http://example.org/fhir/ValueSet/colonoscopy'

context Patient

define "HadColonoscopy": exists ([Procedure: "Colonoscopy"])

In [ ]:
def answer(path):
    frame = cql().exec("HadColonoscopy")
    print(f"{path:22} {frame.filter('`__val`').count()} of {frame.count()} patients had a colonoscopy")

use_resolvers()
answer("environment tables")

**Expected output:** `environment tables     189 of 4000 patients had a colonoscopy`.

The same question over the files in the volumes: one `<Type>.ndjson` per FHIR type, read with `read_files` at each
run, and one FHIR `ValueSet` per file, its codes in `expansion.contains`. The engine expands nothing, so a value set
with only a `compose` is refused, naming it.

In [ ]:
from cql_databricks.resolvers import FolderTerminologyResolver, VolumeDataResolver

use_resolvers(data=VolumeDataResolver(f"{DATA_FILES}/fhir"),
              terminology=FolderTerminologyResolver(f"{KNOWLEDGE_FILES}/valuesets"))
answer("files in a volume")

**Expected output:** `files in a volume      189 of 4000 patients had a colonoscopy`.

And over DataFrames: the patients reshaped from the same NDJSON (`FhirJson`), and the value sets as rows. A DataFrame
lives in this session only, so nothing computed over one is ever kept.

In [ ]:
from cql_databricks.compiler import FhirJson
from cql_databricks.resolvers import DataFrameDataResolver, DataFrameTerminologyResolver

fhir = FhirJson(spark.read.text(f"{DATA_FILES}/fhir/*.ndjson"), column="value")
frames = {t: fhir.to_df(spark, t) for t in ("Patient", "Procedure")}
use_resolvers(data=DataFrameDataResolver(frames, name="paths"),
              terminology=DataFrameTerminologyResolver(spark.table(f"{environment.schema('knowledge')}.valueset"),
                                                       name="paths"))
answer("DataFrames")

**Expected output:** `DataFrames             189 of 4000 patients had a colonoscopy`.

## 2. Knowledge: the environment, or a folder
A library run by name comes from the environment's knowledge, its includes too (`MeasureCommon`, `FHIRHelpers`). A
folder of `.cql` files serves it instead, in a Volume or in the workspace, such as a Git Folder; an include the
folder does not hold is the environment's. The workspace folder here is the `measures/` folder published beside
this notebook. The `%%cql` cell
above is the third path: a library declared in the notebook.

**Expected output:** each population of `ColorectalScreening` with the number of patients in it, the same from all
three: the environment, the Volume and the workspace folder.

In [ ]:
import pathlib

from cql_databricks.resolvers import FolderKnowledgeResolver

_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
MEASURES = pathlib.Path("/Workspace") / pathlib.PurePosixPath(_nb).parent.parent.relative_to("/") / "measures"

def populations():
    frames = cql(library="ColorectalScreening").with_measurement_period("2026-01-01", "2026-12-31").exec()
    return {name: frame.filter("`__val`").count() for name, frame in frames.items()}

use_resolvers()
from_environment = populations()
use_resolvers(knowledge=FolderKnowledgeResolver(f"{KNOWLEDGE_FILES}/libraries"))
from_volume = populations()
use_resolvers(knowledge=FolderKnowledgeResolver(str(MEASURES)))
from_workspace = populations()
print(f"{'':24} {'environment':>11} {'volume':>7} {'workspace':>9}")
for name, count in from_environment.items():
    print(f"{name:24} {count:11} {from_volume[name]:7} {from_workspace[name]:9}")
assert from_volume == from_environment == from_workspace, "the folders hold the environment's libraries"

## 3. Results: kept for this session
`exec()` reads results already computed, and keeps nothing. `materialize` keeps what it computes in a result store.
`SessionResultStore` keeps it in this session's temporary tables: the next run in the session reuses it, nothing
outside the session sees it, and it goes when the session ends (`CLIENT-SESSION-RESULTS`). Layer it with `Overlay`
to reuse the environment's results first:

| Reuse | `results=` |
|---|---|
| Official, then this session's | `Overlay(SessionResultStore(spark=spark), official)` |
| Your own, then this session's | `Overlay(SessionResultStore(spark=spark), own)` |
| Your own, then official, kept in `draft_results` | `EnvironmentResolver(...).draft_store(environment)` |

**Expected output:** the first run computes both populations, the second reuses them.

In [ ]:
from cql_databricks.result import SessionResultStore, materialize

use_resolvers()
query = cql(library="ColorectalScreening").with_measurement_period("2026-01-01", "2026-12-31")
session = SessionResultStore(spark=spark)
for attempt in ("first run", "second run"):
    kept = materialize(query, "Denominator", "Numerator", spark=spark, results=session)
    print(f"{attempt:11}", {name: kept.results[name].decision for name in ("Denominator", "Numerator")})

---
### Where next
**Getting Started / Mapping Your Data** maps tables of your own to FHIR and checks the mapping. Back to
**Start Here** for the index.

In [ ]:
# Every cell above ran, so the notebook is green. The narrative states each expected result; a compile or runtime
# failure fails its own cell.
use_resolvers()
dbutils.notebook.exit(json.dumps({"ok": True, "notebook": "getting_started/03_data_and_knowledge_resolution"}))